# 🔍 RAG Pipeline — Step-by-Step Walkthrough
### LangChain + ChromaDB + Groq (LLaMA 3)
This notebook explains every step of the RAG pipeline interactively.

## Step 1: Install Dependencies

In [ ]:
!pip install langchain langchain-community langchain-groq chromadb sentence-transformers pypdf streamlit -q

## Step 2: Set Your Groq API Key
Get a free key at https://console.groq.com

In [ ]:
import os
os.environ['GROQ_API_KEY'] = 'your_groq_api_key_here'

## Step 3: Load Documents

In [ ]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader('../docs/supply_chain_overview.txt')
documents = loader.load()
print(f'Loaded {len(documents)} document(s)')
print(f'Preview: {documents[0].page_content[:300]}')

## Step 4: Chunk the Documents
We split into smaller overlapping chunks so the retriever can find precise answers.

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = splitter.split_documents(documents)
print(f'Created {len(chunks)} chunks')
print(f'\nSample chunk:\n{chunks[0].page_content}')

## Step 5: Embed + Store in ChromaDB
HuggingFace embeddings run locally — no API key needed.

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
vectorstore = Chroma.from_documents(chunks, embedding=embeddings, persist_directory='./chroma_db')
print('Vector store created and persisted!')

## Step 6: Build QA Chain with Groq LLM

In [ ]:
from langchain_groq import ChatGroq
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

llm = ChatGroq(model_name='llama3-8b-8192', temperature=0.2)

prompt_template = """Use the context to answer the question.
If unsure, say you don't have enough information.

Context: {context}
Question: {question}
Answer:"""

prompt = PromptTemplate(template=prompt_template, input_variables=['context', 'question'])

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectorstore.as_retriever(search_kwargs={'k': 3}),
    chain_type_kwargs={'prompt': prompt},
    return_source_documents=True
)
print('QA Chain ready!')

## Step 7: Ask Questions!

In [ ]:
questions = [
    'What is terminal throughput?',
    'How does RAG help in supply chain management?',
    'What is yard management?'
]

for q in questions:
    result = qa_chain.invoke({'query': q})
    print(f'Q: {q}')
    print(f'A: {result["result"]}')
    print('-' * 60)